In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2
from IPython.display import Image, display

# Preparación de los datos

## Introducción

### Objetivos

- Comprender la importancia de una adecuada preparación de los datos.

- Identificar los principales desafíos al trabajar con datos.

- Conectar los pasos de Data-Cleaning y Standardization con la toma de decisiones.

### Breve repaso del ciclo de vida de un proyecto de análisis de datos:

1. Comprensión del problema de negocio

2. Recolección y comprensión de los datos

3. Limpieza y preparación

4. Análisis y modelado

5. Comunicación de resultados

### Contexto:

- La calidad de los datos tienen un gran impacto a la hora de realizar un análisis descriptivo y obtener un buen desempeño del modelo.

- Garbage in, garbage out.

- Para llegar a buenos resultados se hace necesario corregir inconsistencias, errores e información irrelevante.

### ¿Qué es data processing?

- Procesar la data inicial para análisis posteriores así como tareas de modelado.

- Paso preliminar al analisis de datos.

- La idea es procesar la data en un formato que pueda ser usado tareas como:
  * Analisis de datos
  * Machine Learning
  * Sciencia de datos
  * IA

# Pasos para el procesamiento de datos

## 1. Limpieza de datos

Corregir errores e inconsistencias con la finalidad de tener data lo más precisa y completa posible.

*Valores perdidos*
  
  - Imputación por media o moda
  - Eliminar registros
  - Usar modelos predictivos para llenar valores faltantes

*Remover duplicados*

  - Eliminar registros duplicados para garantizar que cada entrada sea única y relevante.

*Corregir inconsistencias en formatos*

  - Estandarizar formatos como fechas, mayúsculas o minúsculas en textos.



In [2]:
data = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'cliente': ['Juan P.', 'Maria G', 'Carlos S.', 'Juan P.', None],
    'edad': [28, 34, None, 28, 22],
    'precio': [100.5, None, 85.3, 100.5, 50.0],
    'fecha': ['2023/12/01', '2023/12/02', '2023/12/01', '2023/12/01', '2023/12/03']
	})
data

,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.5,2023/12/01
1,2,Maria G,34.0,NaN,2023/12/02
2,3,Carlos S.,NaN,85.3,2023/12/01
3,4,Juan P.,28.0,100.5,2023/12/01
4,5,None,22.0,50.0,2023/12/03


In [7]:
imputer = SimpleImputer(strategy='mean')
data[['edad', 'precio']] = imputer.fit_transform(data[['edad', 'precio']])

In [ ]:
# Llenar valores nulos
edad_mean = data['edad'].mean()
precio_mean = data['precio'].mean()

data['edad'] = data['edad'].fillna(edad_mean)
data['precio'] = data['precio'].fillna(precio_mean)

'''
# se podría hacer también de la siguiente manera:
imputer = SimpleImputer(strategy='mean')
data[['edad', 'precio']] = imputer.fit_transform(data[['edad', 'precio']])
'''

data

,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.500,2023/12/01
1,2,Maria G,34.0,84.075,2023/12/02
2,3,Carlos S.,28.0,85.300,2023/12/01
3,4,Juan P.,28.0,100.500,2023/12/01
4,5,None,22.0,50.000,2023/12/03


#### ¿Le ven sentido a imputar por la media? ¿Se les ocurre otra forma de solucionar este problema de falta de datos?

In [13]:
# Eliminar valores duplicados
data = data.drop_duplicates(subset=data.columns[1:])
data

,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.500,2023/12/01
1,2,Maria G,34.0,84.075,2023/12/02
2,3,Carlos S.,28.0,85.300,2023/12/01
4,5,None,22.0,50.000,2023/12/03


In [14]:
# transformando fecha a tipo date_time
data['fecha'] = pd.to_datetime(data['fecha'], errors='coerce')

# llenando nombre faltante de clientes
data['cliente'] = data['cliente'].fillna('Sin Nombre')

data

,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.500,2023-12-01
1,2,Maria G,34.0,84.075,2023-12-02
2,3,Carlos S.,28.0,85.300,2023-12-01
4,5,Sin Nombre,22.0,50.000,2023-12-03


In [15]:
data.dtypes

,0
id,int64
cliente,object
edad,float64
precio,float64
fecha,datetime64[ns]


## 2. Data Integration

Combinar data de múltiples fuentes

In [16]:
data_clientes = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Juan P.', 'Maria G.', 'Carlos S.', 'Fernando M.'],
    'age': [28, 34, 29, 20]
})
data_clientes

,customer_id,name,age
0,1,Juan P.,28
1,2,Maria G.,34
2,3,Carlos S.,29
3,4,Fernando M.,20


In [17]:
data_ventas = pd.DataFrame({
    'customer_id': [1, 3, 4, 5],
    'purchase_amount': [100.5, 85.3, 45.0, 35.6],
    'purchase_date': ['2023-12-01', '2023-12-02', '2023-12-03', '2023-12-03']
})
data_ventas

,customer_id,purchase_amount,purchase_date
0,1,100.5,2023-12-01
1,3,85.3,2023-12-02
2,4,45.0,2023-12-03
3,5,35.6,2023-12-03


In [19]:
data_paises = pd.DataFrame({
    'user_id': [1, 2, 3, 4],
    'country': ['US', 'UK', 'MX', 'AR']
})
data_paises

,user_id,country
0,1,US
1,2,UK
2,3,MX
3,4,AR


In [20]:
data_clientes

,customer_id,name,age
0,1,Juan P.,28
1,2,Maria G.,34
2,3,Carlos S.,29
3,4,Fernando M.,20


In [21]:
data_ventas

,customer_id,purchase_amount,purchase_date
0,1,100.5,2023-12-01
1,3,85.3,2023-12-02
2,4,45.0,2023-12-03
3,5,35.6,2023-12-03


In [25]:
# haciendo join de los datos con la llave 'customer_id'
merged_data = pd.merge(data_clientes, data_ventas, on='customer_id', how='right')

merged_data

,customer_id,name,age,purchase_amount,purchase_date
0,1,Juan P.,28.0,100.5,2023-12-01
1,3,Carlos S.,29.0,85.3,2023-12-02
2,4,Fernando M.,20.0,45.0,2023-12-03
3,5,NaN,NaN,35.6,2023-12-03


Tipos de join

In [ ]:
url = "https://datacomy.com/data_analysis/pandas/merge/types-of-joins.png"
display(Image(url=url, width=500))


In [27]:
data_clientes

,customer_id,name,age
0,1,Juan P.,28
1,2,Maria G.,34
2,3,Carlos S.,29
3,4,Fernando M.,20


In [28]:
data_paises

,user_id,country
0,1,US
1,2,UK
2,3,MX
3,4,AR


In [45]:
# ejemplo con columnas diferentes
merged_data = pd.merge(data_clientes, data_paises, left_on='customer_id', right_on='user_id', how='inner')
merged_data = merged_data.drop(columns=['user_id'])
merged_data

,customer_id,name,age,country
0,1,Juan P.,28,US
1,2,Maria G.,34,UK
2,3,Carlos S.,29,MX
3,4,Fernando M.,20,AR


In [48]:
data_nuevos_nombres = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Beto', 'Ana', 'Carlos', 'Daniela']
})
data_nuevos_nombres

,customer_id,name
0,1,Beto
1,2,Ana
2,3,Carlos
3,4,Daniela


In [49]:
data_clientes

,customer_id,name,age
0,1,Juan P.,28
1,2,Maria G.,34
2,3,Carlos S.,29
3,4,Fernando M.,20


In [50]:
data_nuevos_nombres

,customer_id,name
0,1,Beto
1,2,Ana
2,3,Carlos
3,4,Daniela


In [52]:
# ejemplo usando sufijos
merged_data = pd.merge(data_clientes, data_nuevos_nombres, on='customer_id', suffixes=('_old', '_new'))
merged_data

,customer_id,name_old,age,name_new
0,1,Juan P.,28,Beto
1,2,Maria G.,34,Ana
2,3,Carlos S.,29,Carlos
3,4,Fernando M.,20,Daniela


In [55]:
data_ventas_ciudad = pd.DataFrame({
    'city': ['NY', 'NY', 'LA', 'LA', 'TX'],
    'year': [2020, 2021, 2020, 2021, 2020],
    'sales': [100, 150, 200, 180, 220]
})
data_ventas_ciudad

,city,year,sales
0,NY,2020,100
1,NY,2021,150
2,LA,2020,200
3,LA,2021,180
4,TX,2020,220


In [56]:
data_ganancias_ciudad = pd.DataFrame({
    'city': ['NY', 'NY', 'LA', 'TX', 'TX'],
    'year': [2020, 2022, 2020, 2020, 2021],
    'profit': [30, 70, 50, 40, 60]
})
data_ganancias_ciudad

,city,year,profit
0,NY,2020,30
1,NY,2022,70
2,LA,2020,50
3,TX,2020,40
4,TX,2021,60


In [57]:
# merge usando multiples keys
#merged_multi = pd.merge(data_ventas_ciudad, data_ganancias_ciudad, on=['city', 'year'], how='outer')
merged_multi = pd.merge(data_ventas_ciudad, data_ganancias_ciudad, left_on=['city', 'year'], right_on=['city', 'year'], how='outer')
merged_multi


,city,year,sales,profit
0,LA,2020,200.0,50.0
1,LA,2021,180.0,NaN
2,NY,2020,100.0,30.0
3,NY,2021,150.0,NaN
4,NY,2022,NaN,70.0
5,TX,2020,220.0,40.0
6,TX,2021,NaN,60.0


## 3. Transformación de datos

Convertir la data en un formato adecuado para análisis.

- **Escalado y normalización:** Importante para algoritmos que dependen de métricas de distancia.
- **Codificación de datos:** Convertir variables categóricas en valores numéricos mediante one-hot encoding o label encoding.



In [58]:
data = pd.DataFrame({
    'client_category': ['A', 'B', 'A', 'C', 'B'],
    'client_age': [20, 30, 20, 40, 30]
	})
data

,client_category,client_age
0,A,20
1,B,30
2,A,20
3,C,40
4,B,30


In [59]:
# estandarizacion de datos
mu = np.mean(data['client_age'])
sigma = np.std(data['client_age'])
data['standardized_client_age'] = (data['client_age'] - mu) / sigma
data

,client_category,client_age,standardized_client_age
0,A,20,-1.069045
1,B,30,0.267261
2,A,20,-1.069045
3,C,40,1.603567
4,B,30,0.267261


In [60]:
# metodo alternativo
scaler = StandardScaler()
data['standardized_client_age'] = scaler.fit_transform(data[['client_age']])
data

,client_category,client_age,standardized_client_age
0,A,20,-1.069045
1,B,30,0.267261
2,A,20,-1.069045
3,C,40,1.603567
4,B,30,0.267261


In [61]:
# normalizacion min-max
min_age = np.min(data['client_age'])
max_age = np.max(data['client_age'])
data['normalized_client_age'] = (data['client_age'] - min_age) / (max_age - min_age)
data

,client_category,client_age,standardized_client_age,normalized_client_age
0,A,20,-1.069045,0.0
1,B,30,0.267261,0.5
2,A,20,-1.069045,0.0
3,C,40,1.603567,1.0
4,B,30,0.267261,0.5


Al final todas las tareas de Machine Learning no son mas que definir una funcion $f(\mathbf{x};  \mathbf{w})$, donde $\mathbf{x}$ son las features de entrada, y $\mathbf{w}$ son un conjunto de parametros que se definen mediante un proceso de optimizacion basado en la data de entrenamiento. Cuando los pesos del modelo $\mathbf{w}$ estan definidos decimos que el modelo esta entrenado.

Es por esto que es importante entender que debemos numerizar de alguna forma aquellas variables categoricas que tenemos en nuestro dataset para poder ser procesados.

In [62]:
w = [1, 2]
X = np.array([
    [1., 0.5],
    [1., 0.0],
    [1., 1.0],
    [1., 5.0],
    [1., 8.0]
])
y = np.matmul(X, w)
y

array([ 2.,  1.,  3., 11., 17.])

In [64]:
# label encoder
label_encoder = LabelEncoder()
data['client_category_encoded'] = label_encoder.fit_transform(data['client_category'])
data

,client_category,client_age,standardized_client_age,normalized_client_age,client_category_encoded
0,A,20,-1.069045,0.0,0
1,B,30,0.267261,0.5,1
2,A,20,-1.069045,0.0,0
3,C,40,1.603567,1.0,2
4,B,30,0.267261,0.5,1


In [65]:
# one-hot encoding
onehot_encoder = OneHotEncoder(sparse_output=False)
encoded_data = pd.DataFrame(
    onehot_encoder.fit_transform(data[['client_category']]),
    columns=onehot_encoder.get_feature_names_out(['client_category'])
)

data = pd.concat([data, encoded_data], axis=1)

data

,client_category,client_age,standardized_client_age,normalized_client_age,client_category_encoded,client_category_A,client_category_B,client_category_C
0,A,20,-1.069045,0.0,0,1.0,0.0,0.0
1,B,30,0.267261,0.5,1,0.0,1.0,0.0
2,A,20,-1.069045,0.0,0,1.0,0.0,0.0
3,C,40,1.603567,1.0,2,0.0,0.0,1.0
4,B,30,0.267261,0.5,1,0.0,1.0,0.0


# Ejemplo práctico

In [66]:
# preparing data
traveler = pd.DataFrame({
  'user_id': [136, 284, 101, 529, 800, 823],
  'age': [None, 38, 30, 43, 49, 28],
  'name': ["Ana", "Jose", "Tomas", "Viviana", "Carolina", "Katerine"]
})

travel = pd.DataFrame({
  'user_id': [101, 284, 136, 800, 101, 800, 823, 529, 284],
  'date_of_journey': [
    '2018-01-16',
    '2017-07-13',
    '2019-10-10',
    '2018/03/20',
    '2019-12-24',
    '2017-10-17',
    '2016/11/02',
    '2019/09/14',
    '2019-08-07'
  ],
  'duration': [3, 3,2,4,2,3.5,4.5, 2.5, 4.3],
  'destination': [
    "Costa Rica",
    "colombia",
    "Colombia",
    "Costa_Rica",
    "Colombia/",
    "Colombia",
    "Costa Rica",
    "Colombia",
    "Costa_Rica"
  ],
  'cost': [None, 775, 587, 913, 1333, 825, 1046, 613, 970],
  'currency': [None, 'EUR', 'USD', 'USD', 'USD','EUR', 'USD', 'USD', 'USD']
})

In [67]:
traveler

,user_id,age,name
0,136,NaN,Ana
1,284,38.0,Jose
2,101,30.0,Tomas
3,529,43.0,Viviana
4,800,49.0,Carolina
5,823,28.0,Katerine


In [68]:
travel

,user_id,date_of_journey,duration,destination,cost,currency
0,101,2018-01-16,3.0,Costa Rica,NaN,None
1,284,2017-07-13,3.0,colombia,775.0,EUR
2,136,2019-10-10,2.0,Colombia,587.0,USD
3,800,2018/03/20,4.0,Costa_Rica,913.0,USD
4,101,2019-12-24,2.0,Colombia/,1333.0,USD
5,800,2017-10-17,3.5,Colombia,825.0,EUR
6,823,2016/11/02,4.5,Costa Rica,1046.0,USD
7,529,2019/09/14,2.5,Colombia,613.0,USD
8,284,2019-08-07,4.3,Costa_Rica,970.0,USD


In [71]:
data = pd.merge(traveler, travel, on='user_id', how='outer')
data

,user_id,age,name,date_of_journey,duration,destination,cost,currency
0,101,30.0,Tomas,2018-01-16,3.0,Costa Rica,NaN,None
1,101,30.0,Tomas,2019-12-24,2.0,Colombia/,1333.0,USD
2,136,NaN,Ana,2019-10-10,2.0,Colombia,587.0,USD
3,284,38.0,Jose,2017-07-13,3.0,colombia,775.0,EUR
4,284,38.0,Jose,2019-08-07,4.3,Costa_Rica,970.0,USD
5,529,43.0,Viviana,2019/09/14,2.5,Colombia,613.0,USD
6,800,49.0,Carolina,2018/03/20,4.0,Costa_Rica,913.0,USD
7,800,49.0,Carolina,2017-10-17,3.5,Colombia,825.0,EUR
8,823,28.0,Katerine,2016/11/02,4.5,Costa Rica,1046.0,USD


In [76]:
data['date_of_journey'] = data['date_of_journey'].str.replace('/', '-')
data['date_of_journey'] = pd.to_datetime(data['date_of_journey'])
data

,user_id,age,name,date_of_journey,duration,destination,cost,currency
0,101,30.0,Tomas,2018-01-16,3.0,Costa Rica,NaN,None
1,101,30.0,Tomas,2019-12-24,2.0,Colombia/,1333.0,USD
2,136,NaN,Ana,2019-10-10,2.0,Colombia,587.0,USD
3,284,38.0,Jose,2017-07-13,3.0,colombia,775.0,EUR
4,284,38.0,Jose,2019-08-07,4.3,Costa_Rica,970.0,USD
5,529,43.0,Viviana,2019-09-14,2.5,Colombia,613.0,USD
6,800,49.0,Carolina,2018-03-20,4.0,Costa_Rica,913.0,USD
7,800,49.0,Carolina,2017-10-17,3.5,Colombia,825.0,EUR
8,823,28.0,Katerine,2016-11-02,4.5,Costa Rica,1046.0,USD


In [85]:
x = np.arange(1, 20)
x

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19])

In [90]:
mask = x > 5
mask

array([False, False, False, False, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [91]:
x[mask]

array([ 6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])

In [92]:
x[x > 5]

array([ 6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])

In [95]:
data['destination'].str.lower().str.contains('costa')

,destination
0,True
1,False
2,False
3,False
4,True
5,False
6,True
7,False
8,True


In [97]:
data.loc[data['destination'].str.lower().str.contains('colombia'), 'destination'] = 'Colombia'
data.loc[data['destination'].str.lower().str.contains('costa'), 'destination'] = 'Costa Rica'

In [101]:
conversion_rate = 1.16 # USD/EUR

In [105]:
data.loc[data['currency'] == 'EUR', 'cost'] = data.loc[data['currency'] == 'EUR', 'cost'] * conversion_rate

In [107]:
data['currency'] = 'USD'

In [113]:
data.loc[data['destination'] == 'Costa Rica', 'cost'].mean()

np.float64(976.3333333333334)

In [115]:
cost_mean_costa_rica = data[data['destination'] == 'Costa Rica']['cost'].mean()

In [118]:
data['cost'].fillna(cost_mean_costa_rica, inplace=True)
data

/tmp/ipykernel_8211/4062607562.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['cost'].fillna(cost_mean_costa_rica, inplace=True)


,user_id,age,name,date_of_journey,duration,destination,cost,currency
0,101,30.0,Tomas,2018-01-16,3.0,Costa Rica,976.333333,USD
1,101,30.0,Tomas,2019-12-24,2.0,Colombia,1333.000000,USD
2,136,NaN,Ana,2019-10-10,2.0,Colombia,587.000000,USD
3,284,38.0,Jose,2017-07-13,3.0,Colombia,899.000000,USD
4,284,38.0,Jose,2019-08-07,4.3,Costa Rica,970.000000,USD
5,529,43.0,Viviana,2019-09-14,2.5,Colombia,613.000000,USD
6,800,49.0,Carolina,2018-03-20,4.0,Costa Rica,913.000000,USD
7,800,49.0,Carolina,2017-10-17,3.5,Colombia,957.000000,USD
8,823,28.0,Katerine,2016-11-02,4.5,Costa Rica,1046.000000,USD


In [119]:
age_mean = data['age'].mean()
data['age'] = data['age'].fillna(age_mean)

In [124]:
data['age'] = data['age'].astype('int')
data['cost'] = np.round(data['cost'], 2)

data

,user_id,age,name,date_of_journey,duration,destination,cost,currency
0,101,30,Tomas,2018-01-16,3.0,Costa Rica,976.33,USD
1,101,30,Tomas,2019-12-24,2.0,Colombia,1333.00,USD
2,136,38,Ana,2019-10-10,2.0,Colombia,587.00,USD
3,284,38,Jose,2017-07-13,3.0,Colombia,899.00,USD
4,284,38,Jose,2019-08-07,4.3,Costa Rica,970.00,USD
5,529,43,Viviana,2019-09-14,2.5,Colombia,613.00,USD
6,800,49,Carolina,2018-03-20,4.0,Costa Rica,913.00,USD
7,800,49,Carolina,2017-10-17,3.5,Colombia,957.00,USD
8,823,28,Katerine,2016-11-02,4.5,Costa Rica,1046.00,USD


# House Prices Dataset

In [ ]:
!wget https://raw.githubusercontent.com/nicolasggiraldo/especializacion-udem-2025-02/refs/heads/main/data/house_prices_dataset/data_description.txt
!wget https://raw.githubusercontent.com/nicolasggiraldo/especializacion-udem-2025-02/refs/heads/main/data/house_prices_dataset/train.csv
!wget !wget https://raw.githubusercontent.com/nicolasggiraldo/especializacion-udem-2025-02/refs/heads/main/data/house_prices_dataset/test.csv
!mkdir -p house_price_data
!mv *.csv house_price_data/
!mv data_description.txt house_price_data/

--2026-04-19 17:24:23--  https://raw.githubusercontent.com/nicolasggiraldo/especializacion-udem-2025-02/refs/heads/main/data/house_prices_dataset/data_description.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13370 (13K) [text/plain]
Saving to: ‘data_description.txt’

data_description.tx 100%[===================>]  13.06K  --.-KB/s    in 0s      

2026-04-19 17:24:23 (25.7 MB/s) - ‘data_description.txt’ saved [13370/13370]

--2026-04-19 17:24:23--  https://raw.githubusercontent.com/nicolasggiraldo/especializacion-udem-2025-02/refs/heads/main/data/house_prices_dataset/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubuserc

In [ ]:
!cat house_price_data/data_description.txt

MSSubClass: Identifies the type of dwelling involved in the sale.	

        20	1-STORY 1946 & NEWER ALL STYLES
        30	1-STORY 1945 & OLDER
        40	1-STORY W/FINISHED ATTIC ALL AGES
        45	1-1/2 STORY - UNFINISHED ALL AGES
        50	1-1/2 STORY FINISHED ALL AGES
        60	2-STORY 1946 & NEWER
        70	2-STORY 1945 & OLDER
        75	2-1/2 STORY ALL AGES
        80	SPLIT OR MULTI-LEVEL
        85	SPLIT FOYER
        90	DUPLEX - ALL STYLES AND AGES
       120	1-STORY PUD (Planned Unit Development) - 1946 & NEWER
       150	1-1/2 STORY PUD - ALL AGES
       160	2-STORY PUD - 1946 & NEWER
       180	PUD - MULTILEVEL - INCL SPLIT LEV/FOYER
       190	2 FAMILY CONVERSION - ALL STYLES AND AGES

MSZoning: Identifies the general zoning classification of the sale.
		
       A	Agriculture
       C	Commercial
       FV	Floating Village Residential
       I	Industrial
       RH	Residential High Density
       RL	Residential Low Density
       RP	Residential Low Density Park 
       RM

In [ ]:
df = pd.read_csv('house_price_data/train.csv')
print(df.shape)
df.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
